In [1]:
#Install if required
# pip install tensorflow-datasets

# =========================================================
# 1. IMPORT LIBRARIES
# =========================================================
# Core Python + ML libraries
import os
import json

# TensorFlow + datasets
import tensorflow as tf
import tensorflow_datasets as tfds

# Visualization
import matplotlib.pyplot as plt

# Keras modules
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input



2026-03-23 20:35:28.997041: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-23 20:35:29.019698: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-23 20:35:29.019715: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-23 20:35:29.020325: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-23 20:35:29.024104: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# =========================================================
# 2. GPU MEMORY SAFETY (VERY IMPORTANT 🔥)
# =========================================================
# Prevent TensorFlow from grabbing all GPU memory at once
# This avoids OOM (Out Of Memory) issues

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)



# =========================================================
# 3. CONFIGURATION
# =========================================================
# Image size for models
IMG_SIZE = (160, 160)   # smaller than 224 → saves memory

# Batch size (reduce if OOM occurs)
BATCH_SIZE = 16

# Performance optimization
AUTOTUNE = tf.data.AUTOTUNE

# Directory structure
DATA_DIR = "./datasets"
MODEL_DIR = "./saved_models"
PLOT_DIR = "./plots"

# Create directories if they don't exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)



# =========================================================
# 4. LOAD DATASET (TFDS handles download automatically)
# =========================================================
(train_ds, val_ds), info = tfds.load(
    'cats_vs_dogs',
    split=['train[:75%]', 'train[75%:]'],  # 75/25 split
    as_supervised=True,                   # returns (image, label)
    with_info=True,
    data_dir=DATA_DIR
)

2026-03-23 20:35:47.437291: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-23 20:35:47.460449: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-23 20:35:47.462735: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [3]:
# =========================================================
# 5. DATA AUGMENTATION (helps generalization)
# =========================================================
# Applied only during training

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])



# =========================================================
# 6. PREPROCESSING FUNCTIONS
# =========================================================

# For CNN (simple normalization)
def preprocess_cnn(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = image / 255.0  # normalize to [0,1]
    return image, label

# For VGG16 (special preprocessing required)
def preprocess_vgg(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = preprocess_input(image)  # required for pretrained models
    return image, label




In [5]:
type(train_ds)

tensorflow.python.data.ops.prefetch_op._PrefetchDataset